# Progressive Teaching Workflow Notebook

这个 notebook 是一个渐进式教学版本。它不是最短示例，而是带你从“先跑通、看懂输出”，逐步走到“当前推荐的正式 MACE 用法”和“困难异质支撑 case 的处理方式”。

你会在这里完成三件事：

- 先理解这个 repo 的主线：`slab -> surface/site -> pose -> basin -> node`
- 学会当前最推荐的高层生产入口：`generate_adsorption_ensemble(...)`
- 学会什么时候需要升级到显式 `run_adsorption_workflow(...)` 的困难 case 配方

默认叙事以生产环境为主：`mace_les + CUDA + MACE model`。
如果你暂时没有 GPU 或模型，本 notebook 也会告诉你如何降级到概念验证路径。


## Part 0. Workflow Map

先建立一个简单心智模型。这个项目不是固定模板的 `top/bridge/hollow` 放置器，而是：

1. 从 slab 自动识别外表面
2. 枚举 surface primitives
3. 在 primitive 局部坐标系上做 pose sampling
4. 用 relax 把初猜映射到最终 basin
5. 把 basin 规范化成 reaction-ready node

后面每一节都在回答一个更具体的问题：

- 最小入口该怎么用
- 正式生产配置该怎么写
- 结果目录到底该先看什么
- 异质 cluster-on-slab case 为什么要做 case-scoped 扩展


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import json
import os

from ase import Atoms
from ase.build import fcc111, fcc211, molecule

from adsorption_ensemble.relax import IdentityRelaxBackend, MACEBatchRelaxBackend, MaceRelaxConfig
from adsorption_ensemble.workflows import (
    DEFAULT_MACE_HEAD_NAME,
    DEFAULT_MACE_MODEL_PATH,
    generate_adsorption_ensemble,
    make_adsorption_workflow_config,
    make_default_surface_preprocessor,
    make_sampling_schedule,
    run_adsorption_workflow,
)


## Part 1. Environment Check

先判断你现在是不是在推荐环境里。

这个检查主要回答 4 个问题：

- `adsorption_ensemble` 是否可导入
- `torch` / CUDA 是否可用
- MACE 模型路径是否存在
- 本 notebook 接下来该走生产路径，还是先走降级路径

如果这里显示 GPU 或模型不可用，也不用停。后面前半段仍然可以跑，只是它更适合理解 workflow，不适合做物理结论。


In [ ]:
env_status = {
    "adsorption_ensemble": importlib.util.find_spec("adsorption_ensemble") is not None,
    "ase": importlib.util.find_spec("ase") is not None,
    "torch": importlib.util.find_spec("torch") is not None,
}

cuda_available = False
cuda_device_name = None
model_path = Path(os.environ.get("AE_MACE_MODEL_PATH", DEFAULT_MACE_MODEL_PATH))
model_exists = model_path.exists()

if env_status["torch"]:
    import torch

    cuda_available = bool(torch.cuda.is_available())
    if cuda_available:
        cuda_device_name = torch.cuda.get_device_name(0)

use_mace_backend = bool(env_status["torch"] and cuda_available and model_exists)

print("Environment status")
print(
    json.dumps(
        {
            "imports": env_status,
            "cuda_available": cuda_available,
            "cuda_device_name": cuda_device_name,
            "model_path": str(model_path),
            "model_exists": model_exists,
            "use_mace_backend": use_mace_backend,
        },
        ensure_ascii=False,
        indent=2,
    )
)


### How to interpret the environment check

- 如果 `use_mace_backend = true`，你可以直接跟着本 notebook 的生产主线跑。
- 如果 `use_mace_backend = false`，你仍然可以先跑前面的最小流程，但后面的正式生产单元会自动跳过。

这也是本 notebook 的设计原则：

- 前半段先教你理解 API 和输出结构
- 后半段再切到正式 MACE 配置
- 高级 case 默认用显式开关，避免误触长任务


## Part 2. The Minimal High-Level Workflow

这里我们先用最简单、最稳定的高层入口：`generate_adsorption_ensemble(...)`。

这一步的目标不是追求最复杂的物理体系，而是让你先看懂：

- 输入是什么
- `work_dir` 会产生哪些文件
- `result.summary` 和 `result.files` 分别代表什么

这里我故意选了规则表面 `fcc111 + NH3`。这样你能先把注意力放在 workflow 本身，而不是困难几何上。


In [ ]:
quick_work_dir = Path("artifacts/notebook_progressive/minimal_fcc111_nh3")
slab = fcc111("Pt", size=(4, 4, 4), vacuum=12.0)
adsorbate = molecule("NH3")

if use_mace_backend:
    quick_backend = MACEBatchRelaxBackend(
        MaceRelaxConfig(
            model_path=str(model_path),
            device="cuda",
            dtype="float32",
            head_name=DEFAULT_MACE_HEAD_NAME,
            enable_cueq=True,
            strict=True,
            max_edges_per_batch=100000,
        )
    )
    print("Using MACE backend for the minimal run.")
else:
    quick_backend = IdentityRelaxBackend()
    print("Falling back to IdentityRelaxBackend. This is for workflow understanding only.")

quick_result = generate_adsorption_ensemble(
    slab=slab,
    adsorbate=adsorbate,
    work_dir=quick_work_dir,
    placement_mode="anchor_free",
    schedule=make_sampling_schedule("multistage_default"),
    dedup_metric="mace_node_l2",
    signature_mode="provenance",
    basin_relax_backend=quick_backend,
)

quick_result.summary


### What happened here

这一个高层调用，已经把下面这些步骤串起来了：

- surface detection
- primitive enumeration
- pose sampling
- pre-relax selection
- basin building
- node export

你现在最该做的不是马上继续改参数，而是先看这个结果对象到底提供了什么。


In [ ]:
print("summary:")
print(json.dumps(quick_result.summary, ensure_ascii=False, indent=2))

print("\nimportant files:")
for key in [
    "work_dir",
    "site_dictionary_json",
    "pose_pool_extxyz",
    "pose_pool_selected_extxyz",
    "basins_json",
    "nodes_json",
]:
    print(f"- {key}: {quick_result.files.get(key)}")


## Part 3. How to Read the Outputs

很多用户第一次接触这类 workflow，会把注意力都放在“最后有几个 basin”。这不够。

一个更好的阅读顺序是：

1. `site_dictionary.json`
   - 识别到了哪些 primitive / basis sites
2. `pose_pool.extxyz`
   - 原始采样到底放出了多少初猜
3. `pose_pool_selected.extxyz`
   - 哪些初猜真正进入了 basin relax
4. `basins.json`
   - 最终保留了哪些 basin，以及拒绝了什么
5. `nodes.json`
   - 下游 reaction-ready 的规范化 node 是什么

下面我们只做轻量的文本检查，避免 notebook 里一次性展示太多大文件内容。


In [ ]:
site_dict_path = Path(quick_result.files["site_dictionary_json"])
basins_json_path = Path(quick_result.files["basins_json"])
nodes_json_path = Path(quick_result.files["nodes_json"])

site_dict = json.loads(site_dict_path.read_text(encoding="utf-8"))
basins_dict = json.loads(basins_json_path.read_text(encoding="utf-8"))
nodes = json.loads(nodes_json_path.read_text(encoding="utf-8"))

print("site meta:")
print(json.dumps(site_dict.get("meta", {}), ensure_ascii=False, indent=2))

print("\nbasin summary:")
print(json.dumps(basins_dict.get("summary", {}), ensure_ascii=False, indent=2))

print("\nfirst node preview:")
print(json.dumps(nodes[0] if nodes else {}, ensure_ascii=False, indent=2))


### A healthy first read

一个健康的最小 workflow，通常至少应该满足：

- `n_basis_primitives > 0`
- `n_pose_frames > 0`
- `basins.json` 和 `nodes.json` 都已经生成

如果这些最基本条件都没达到，就先别去做更复杂体系。先把最小例子跑通，并确认目录结构和返回对象理解清楚。


## Part 4. Recommended Production Usage

现在进入当前最推荐的正式用法。

这里的主线仍然是高层 API：`generate_adsorption_ensemble(...)`。
只不过我们显式提供 `MACEBatchRelaxBackend`，并使用当前 repo 已验证过的一组生产参数。

我这里选的是 `Pt(211) + CO`，因为它足够接近真实表面化学使用场景，又比困难异质支撑 case 更容易作为生产入口示例。


In [ ]:
if not use_mace_backend:
    print("Skipping the production run because CUDA/MACE/model availability was not detected.")
else:
    prod_work_dir = Path("artifacts/notebook_progressive/production_pt211_co")
    slab_prod = fcc211("Pt", size=(6, 4, 4), vacuum=12.0)
    adsorbate_prod = molecule("CO")

    prod_backend = MACEBatchRelaxBackend(
        MaceRelaxConfig(
            model_path=str(model_path),
            device="cuda",
            dtype="float32",
            head_name=DEFAULT_MACE_HEAD_NAME,
            enable_cueq=True,
            strict=True,
            max_edges_per_batch=100000,
        )
    )

    prod_result = generate_adsorption_ensemble(
        slab=slab_prod,
        adsorbate=adsorbate_prod,
        work_dir=prod_work_dir,
        placement_mode="anchor_free",
        schedule=make_sampling_schedule("multistage_default"),
        dedup_metric="mace_node_l2",
        signature_mode="provenance",
        basin_overrides={
            "dedup_cluster_method": "greedy",
            "mace_dtype": "float64",
            "mace_enable_cueq": False,
            "desorption_min_bonds": 1,
            "energy_window_ev": 2.5,
        },
        basin_relax_backend=prod_backend,
    )

    print(json.dumps(prod_result.summary, ensure_ascii=False, indent=2))


### Why this is the recommended production path

对于大多数常规 slab，这一套已经足够：

- `generate_adsorption_ensemble(...)` 负责高层流程编排
- `make_sampling_schedule("multistage_default")` 提供稳定默认筛选策略
- `MACEBatchRelaxBackend(...)` 提供真正有物理意义的 relax

也就是说，大多数用户并不需要一开始就自己组装 `AdsorptionWorkflowConfig`。
只有在 coverage 明显不够、或体系本身就是困难 heterogeneous support case 时，才需要升级到下一节的显式 workflow 配方。


## Part 5. Reading Production Results

正式生产 run 跑完后，先不要急着只看最低能 basin。

先看下面这些计数，它们能快速告诉你：这个 run 是不是在正常工作。


In [ ]:
if use_mace_backend:
    keys = [
        "n_surface_atoms",
        "n_basis_primitives",
        "n_pose_frames",
        "n_pose_frames_selected_for_basin",
        "n_basins",
        "n_nodes",
    ]
    for key in keys:
        print(f"{key}: {prod_result.summary.get(key)}")
else:
    print("Production summary is unavailable because the production cell was skipped.")


### What suspicious counts look like

你可以先用一个粗糙的经验判断：

- `n_pose_frames` 很少
  - 可能是 site coverage 不够，或 pose sampling 太保守
- `n_pose_frames` 很多，但 `n_pose_frames_selected_for_basin` 很少
  - 可能是 pre-relax 预算太紧，早早把稀有 motif 丢掉了
- `n_pose_frames_selected_for_basin` 不少，但 `n_basins = 0` 或很少
  - 这时要去看 `basins.json` 里的 rejection reasons，而不是盲目继续加采样

这就是为什么我们一直强调：不要只看最后 basin 数。你需要看 coverage 在哪一层掉了。


## Part 6. Escalating to a Difficult Heterogeneous-Support Case

现在进入真正困难的情况：例如 `cluster-on-slab`、异质金属支撑、台阶 + 柔软 support 等。

这类体系常见问题不是“最后去重错了”，而是更早的 coverage 在 upstream 就丢失了。
这时推荐从高层 API 升级到：

- `make_adsorption_workflow_config(...)`
- `run_adsorption_workflow(...)`
- support pre-relax + case-scoped overrides

下面这个 recipe 复现的是我们在 `Pt(211)+Ag4` case 里已经验证过的思路。


In [ ]:
import math
import numpy as np


def build_pt211_ag4_structure() -> Atoms:
    slab = fcc211("Pt", size=(6, 4, 4), vacuum=12.0)
    pos = np.asarray(slab.get_positions(), dtype=float)
    z = pos[:, 2]
    z_top = float(np.max(z))
    top_ids = np.where(z >= (z_top - 0.45))[0]
    center_xy = np.mean(pos[top_ids, :2], axis=0)

    edge = 2.90
    base_r = edge / math.sqrt(3.0)
    tetra_h = math.sqrt(2.0 / 3.0) * edge
    z_base = z_top + 2.35

    cluster_pos = np.array(
        [
            [0.0, base_r, 0.0],
            [-0.5 * edge, -0.5 * base_r, 0.0],
            [0.5 * edge, -0.5 * base_r, 0.0],
            [0.0, 0.0, tetra_h],
        ],
        dtype=float,
    )
    cluster_pos[:, 0] += float(center_xy[0])
    cluster_pos[:, 1] += float(center_xy[1])
    cluster_pos[:, 2] += float(z_base)

    cluster = Atoms("Ag4", positions=cluster_pos)
    out = slab + cluster
    out.set_cell(slab.cell)
    out.set_pbc(slab.get_pbc())
    return out


RUN_ADVANCED_CASE = False


In [ ]:
if not use_mace_backend:
    print("Skipping the advanced-case recipe because CUDA/MACE/model availability was not detected.")
elif not RUN_ADVANCED_CASE:
    print("Advanced case recipe is prepared but not executed. Set RUN_ADVANCED_CASE = True if you want to run it.")
else:
    advanced_case_root = Path("artifacts/notebook_progressive/advanced_pt211_ag4_co")
    advanced_work_dir = advanced_case_root / "workflow"

    slab_advanced_raw = build_pt211_ag4_structure()
    adsorbate_advanced = molecule("CO")

    advanced_backend = MACEBatchRelaxBackend(
        MaceRelaxConfig(
            model_path=str(model_path),
            device="cuda",
            dtype="float32",
            head_name=DEFAULT_MACE_HEAD_NAME,
            enable_cueq=True,
            strict=True,
            max_edges_per_batch=100000,
        )
    )

    relaxed_frames, slab_energies, _ = advanced_backend.relax(
        frames=[slab_advanced_raw.copy()],
        maxf=0.05,
        steps=200,
        work_dir=advanced_case_root / "slab_relax",
    )
    slab_advanced = relaxed_frames[0].copy()

    schedule = make_sampling_schedule("multistage_default")
    schedule.pre_relax_selection.max_candidates = 64

    cfg = make_adsorption_workflow_config(
        work_dir=advanced_work_dir,
        placement_mode="anchor_free",
        single_atom=False,
        exhaustive_pose_sampling=True,
        dedup_metric="mace_node_l2",
        signature_mode="provenance",
        pose_overrides={
            "adaptive_height_fallback": True,
            "adaptive_height_fallback_step": 0.20,
            "adaptive_height_fallback_max_extra": 1.60,
            "adaptive_height_fallback_contact_slack": 0.60,
        },
        basin_overrides={
            "dedup_cluster_method": "greedy",
            "mace_dtype": "float64",
            "mace_enable_cueq": False,
            "desorption_min_bonds": 1,
            "surface_reconstruction_enabled": False,
            "energy_window_ev": 2.5,
        },
    )
    cfg.surface_preprocessor = make_default_surface_preprocessor(
        target_count_mode="off",
        target_surface_fraction=0.25,
    )
    cfg.pre_relax_selection = schedule.pre_relax_selection
    cfg.basin_config.post_relax_selection = schedule.post_relax_selection
    cfg.max_selected_primitives = None

    advanced_result = run_adsorption_workflow(
        slab=slab_advanced,
        adsorbate=adsorbate_advanced,
        config=cfg,
        basin_relax_backend=advanced_backend,
    )

    print(json.dumps(advanced_result.summary, ensure_ascii=False, indent=2))


### Why these advanced-case settings exist

这几个设置要一起理解，而不是零散地记参数：

- `support pre-relax`
  - 先让 bare support 自己稳定下来，避免 adsorption workflow 把 support 大位移误判成别的问题
- `exhaustive_pose_sampling=True`
  - 增大姿态覆盖，避免稀有 motif 在最早期就没被采到
- `schedule.pre_relax_selection.max_candidates = 64`
  - 让进入 basin relax 的名额更宽，避免 upstream coverage loss
- `cfg.max_selected_primitives = None`
  - 不再用固定 primitive cap 提前截断异质位点
- `adaptive_height_fallback=True`
  - 只在困难 case 里打开，用来缓解初猜碰撞导致的覆盖塌缩
- `surface_reconstruction_enabled=False`
  - 这是 case-scoped 的敏感性设置，不建议直接当作全局默认值

核心原则只有一句：困难 case 要做 case-scoped coverage 扩展，而不是仓促改 repo 默认值。


## Part 7. Real Repo Case Studies

上面的高级 recipe 不是凭空设的。repo 里已经有两组正式 run 可以直接对照：

- `Pt(211)+Ag4+C6H6`
  - `352` poses -> `64` selected -> `37` basins
- `Pt(211)+Ag4+CO`
  - `414` poses -> `64` selected -> `36` basins

它们说明了一件很重要的事：

对异质 cluster-on-slab 体系，成熟用法不是“只开默认 preset 就完事”，而是：

- 默认 preset 作为骨架
- support 先 pre-relax
- 再按 case-scoped 的方式扩 coverage

如果你想直接看 repo 内现成结果，可以去这些目录：

- `artifacts/autoresearch/single_cases/pt211_ag4_cluster_c6h6_20260417/`
- `artifacts/autoresearch/single_cases/pt211_ag4_cluster_co_20260418/`
- `runs/20260417-234454-pt211-ag4-c6h6/`
- `runs/20260418-114025-pt211-ag4-co/`


## Part 8. Practical Tuning Guide

如果一个 case 跑得不理想，建议按这个顺序排查，而不是一开始就改内部实现：

1. 先看 `raw_site_dictionary.json` 和 `site_dictionary.json`
   - 位点识别本身是不是就不够
2. 再看 `pose_pool.extxyz`
   - 是不是初猜覆盖就很少
3. 再看 `pose_pool_selected.extxyz` 和 `pre_relax_selection.json`
   - 是不是 pre-relax 筛选把 coverage 丢掉了
4. 最后看 `basins.json` 的 rejected reasons
   - 是 desorption、dissociation，还是别的异常

一个实用的 first-response checklist：

- coverage 明显不足时，先考虑增加 pre-relax budget
- heterogeneous support 上，考虑 `exhaustive_pose_sampling=True`
- soft support 上，先 pre-relax bare support
- 发现只有困难 case 有问题时，保持设置为 case-scoped
- 除非你已经看过多个真实 case，不要轻易修改 repo 全局默认值

到这里，你应该已经能区分三类入口：

- 先理解 API：最小高层 workflow
- 正式生产：`generate_adsorption_ensemble + MACEBatchRelaxBackend`
- 困难 case：`make_adsorption_workflow_config + run_adsorption_workflow` + case-scoped overrides
